In [1]:
import chromadb

In [2]:
# 1. 创建客户端
client = chromadb.Client()

In [3]:
# 2. 创建集合
collection = client.get_or_create_collection(name="test_collection")

## 默认嵌入函数

In [4]:
print(collection.configuration.get("embedding_function"))

## 自定义嵌入函数

In [5]:
from chromadb import EmbeddingFunction, Documents, Embeddings
import numpy as np

class MyEmbeddingFunction(EmbeddingFunction[Documents]):
    # 初始化
    def __init__(self, len) -> None:
        self.len = len
        return

    # 调用方法
    def __call__(self, input: Documents) -> Embeddings:
        # 返回长度为 len 的数据，元素都是input的长度
        return [ np.full(self.len, len(item)) for item in input ]


In [6]:
# 创建集合，指定嵌入函数
collection2 = client.get_or_create_collection(
    name="collection2",
    embedding_function=MyEmbeddingFunction(len=10)
)

In [7]:
# 插入数据
collection2.add(
    ids=["1", "2", "3"],
    documents=[
        "This is the first document",
        "This is the second document",
        "This is the third document",
    ]
)

In [8]:
collection2.peek()

{'ids': ['1', '2', '3'],
 'embeddings': array([[26., 26., 26., 26., 26., 26., 26., 26., 26., 26.],
        [27., 27., 27., 27., 27., 27., 27., 27., 27., 27.],
        [26., 26., 26., 26., 26., 26., 26., 26., 26., 26.]]),
 'documents': ['This is the first document',
  'This is the second document',
  'This is the third document'],
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings'],
 'data': None,
 'metadatas': [None, None, None]}

In [9]:
collection2.peek()['embeddings'].shape

(3, 10)

In [13]:
from chromadb.api.types import Images
import torch
# 真实场景中的嵌入函数
class ImageEmbeddingFunction(EmbeddingFunction[Images]):
    # 初始化，传入自己的嵌入模型
    def __init__(self, model) -> None:
        self.model = model.to('cpu')
        return

    def __call__(self, input: Images) -> Embeddings:
        # 将输入图像input转换为 Tensor
        input_tensor = torch.tensor(np.ndarray(input))
        # 前向传播，得到模型的输出
        with torch.no_grad():
            output = self.model(input_tensor)
        # 将输出转换为ndarray数组，并返回
        return output.numpy()